# UIT DSC 2026 – LegalQA Generation-Only Smoke Test

Notebook ngắn để kiểm tra **Vi-Qwen2-1.5B-RAG có rút gọn/cắt câu trả lời hay không** mà không xây lại BM25, Dense FAISS hoặc Reranker.

Quy trình:

1. Tự tìm `train.json`, public test và `selected-contexts` theo cùng cơ chế của notebook chính.
2. Ưu tiên `submission_v0.json` làm `CONTEXT` nếu tìm thấy; nếu không có thì dùng các đáp án train dài.
3. So sánh prompt chống tóm tắt ở `max_new_tokens=512` và `1024`.
4. Báo `length_ratio`, mức bảo toàn token, boilerplate, từ chối và dấu hiệu chạm giới hạn token.

> Trên Kaggle, hãy **Add Data** giống notebook chính (dataset LegalQA và output model snapshot nếu có). Nếu không có snapshot, bật Internet để tải model từ Hugging Face. Notebook này không đánh giá retrieval; nó chỉ trả lời câu hỏi “generation có đang rút gọn quá mức không?”.


In [1]:
from __future__ import annotations

import importlib.util
import json
import os
import re
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

# ===== CẤU HÌNH NHANH =====
REPO_URL = "https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git"
GENERATOR_MODEL_ID = "AITeamVN/Vi-Qwen2-1.5B-RAG"
TEST_LIMIT = 10                 # 10 ID x 2 cấu hình; tăng lên 20 nếu muốn
RUN_1024_TEST = True            # False nếu chỉ muốn test 512 nhanh hơn
USE_ACTUAL_QUESTIONS = True     # False: dùng câu lệnh kiểm tra chung
MAX_INPUT_TOKENS_512 = 7168
MAX_INPUT_TOKENS_1024 = 6144
REPETITION_PENALTY = 1.05
SEED = 2026

# Ghi đè đường dẫn nếu auto-discovery chọn sai; để None để tự tìm.
DATASET_DIR = None
TRAIN_PATH = None
PUBLIC_PATH = None
CONTEXTS_PATH = None
SOURCE_SUBMISSION_PATH = None       # Tùy chọn: dùng answer của submission làm CONTEXT
COMPARISON_SUBMISSION_PATH = None   # Tùy chọn: so sánh độ rút ngắn với submission khác
MODEL_PATH = None

# Các ID đã thấy bị rút mạnh/từ chối trong v1.
PRIORITY_IDS = ["34235", "62147", "86293", "80189", "135669"]

if Path("/kaggle/working").is_dir():
    PLATFORM = "Kaggle"
    REPO_DIR = Path("/kaggle/working/uit-dsc-2026-task2-legalqa")
    INPUT_ROOT = Path("/kaggle/input")
    WORK_DIR = Path("/kaggle/working/legalqa-run")
    EXPORT_DIR = Path("/kaggle/working")
elif Path("/content").is_dir():
    PLATFORM = "Colab"
    REPO_DIR = Path("/content/uit-dsc-2026-task2-legalqa")
    INPUT_ROOT = Path("/content")
    WORK_DIR = Path("/content/legalqa-run")
    EXPORT_DIR = WORK_DIR
else:
    PLATFORM = "Local"
    REPO_DIR = Path(".").resolve()
    if not (REPO_DIR / "legalqa_baseline").is_dir():
        candidate = REPO_DIR / "UIT_DSC_2026_LegalQA_baseline_v0.1"
        if candidate.is_dir():
            REPO_DIR = candidate
    INPUT_ROOT = REPO_DIR
    WORK_DIR = REPO_DIR / "artifacts"
    EXPORT_DIR = WORK_DIR

HF_CACHE_DIR = WORK_DIR / "hf-cache"
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")

print(f"Platform: {PLATFORM}")
print(f"Input root: {INPUT_ROOT}")
print(f"Repo: {REPO_DIR}")
print(f"Work dir: {WORK_DIR}")
print(f"Output: {EXPORT_DIR}")


Platform: Kaggle
Input root: /kaggle/input
Repo: /kaggle/working/uit-dsc-2026-task2-legalqa
Work dir: /kaggle/working/legalqa-run
Output: /kaggle/working


## 1. Cài thư viện tối thiểu

Chỉ cài generator dependencies; không cài FAISS, embedding hoặc reranker.


In [2]:
required = {
    "transformers": "transformers>=4.40.0",
    "accelerate": "accelerate",
    "sentencepiece": "sentencepiece",
    "pandas": "pandas",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Dependencies already available.")

if not (REPO_DIR / "legalqa_baseline").is_dir():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(f"REPO_DIR tồn tại nhưng không phải project LegalQA: {REPO_DIR}")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import pandas as pd
import torch
from IPython.display import FileLink, Markdown, display
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from legalqa_baseline.generator import ViQwenRAGGenerator

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    for gpu_id in range(torch.cuda.device_count()):
        print(f"  GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}")


Dependencies already available.


Cloning into '/kaggle/working/uit-dsc-2026-task2-legalqa'...


Torch: 2.10.0+cu128
CUDA: True
  GPU 0: Tesla T4
  GPU 1: Tesla T4


## 2. Tự tìm dataset, submission tùy chọn và model snapshot


In [3]:
MODEL_MARKER = ".legalqa_model.json"


def search_roots():
    candidates = [
        Path(DATASET_DIR) if DATASET_DIR is not None else None,
        INPUT_ROOT,
        WORK_DIR,
        REPO_DIR,
        REPO_DIR.parent,
        Path("./data"),
        Path(".").resolve(),
    ]
    seen = set()
    for root in candidates:
        if root is None:
            continue
        root = Path(root)
        if not root.exists():
            continue
        key = str(root.resolve())
        if key not in seen:
            seen.add(key)
            yield root


def resolve_file(label, override, accepted_names, required=True):
    if override is not None:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f"{label} không tồn tại: {path}")
        return path

    accepted = {name.casefold() for name in accepted_names}
    for root in search_roots():
        matches = sorted(
            (path for path in root.rglob("*") if path.is_file() and path.name.casefold() in accepted),
            key=lambda path: (len(path.parts), str(path)),
        )
        if matches:
            return matches[0]
    if required:
        raise FileNotFoundError(f"Không tìm thấy {label}: {sorted(accepted_names)}")
    return None


def resolve_contexts(override):
    if override is not None:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(f"selected-contexts không tồn tại: {path}")
        return path

    for root in search_roots():
        candidates = sorted(
            root.rglob("*selected-contexts*"),
            key=lambda path: (len(path.parts), str(path)),
        )
        for path in candidates:
            if not path.is_dir():
                continue
            nested = path / "selected-contexts"
            if nested.is_dir() and any(nested.glob("context_*.json")):
                return nested
            if any(path.glob("context_*.json")) or any(path.rglob("context_*.json")):
                return path
        for path in candidates:
            if path.is_file() and path.suffix.casefold() == ".zip":
                return path
    return None


def model_snapshot_is_complete(model_dir):
    return (
        (model_dir / "config.json").is_file()
        and (
            any(model_dir.glob("*.safetensors"))
            or any(model_dir.glob("pytorch_model*.bin"))
        )
    )


def discover_generator_snapshot():
    if MODEL_PATH is not None:
        model_path = Path(MODEL_PATH)
        return str(model_path) if model_path.exists() else str(MODEL_PATH)
    for root in search_roots():
        for marker in root.rglob(MODEL_MARKER):
            try:
                payload = json.loads(marker.read_text(encoding="utf-8"))
            except (OSError, ValueError):
                continue
            if payload.get("repo_id") == GENERATOR_MODEL_ID and model_snapshot_is_complete(marker.parent):
                print(f"Tái sử dụng generator snapshot: {marker.parent}")
                return str(marker.parent)
    print("Không thấy snapshot; sẽ tải generator từ Hugging Face.")
    return GENERATOR_MODEL_ID


TRAIN_PATH = resolve_file("train set", TRAIN_PATH, {"train.json"})
PUBLIC_PATH = resolve_file(
    "public test",
    PUBLIC_PATH,
    {"public-official.json", "public-official(1).json", "public_official.json", "public_test.json"},
)
CONTEXTS_PATH = resolve_contexts(CONTEXTS_PATH)
SOURCE_SUBMISSION_PATH = resolve_file(
    "source submission",
    SOURCE_SUBMISSION_PATH,
    {"submission_v0.json"},
    required=False,
)
COMPARISON_SUBMISSION_PATH = resolve_file(
    "comparison submission",
    COMPARISON_SUBMISSION_PATH,
    {"submission_v1.json", "submission_v1(1).json", "submission_rag.json"},
    required=False,
)
GENERATOR_MODEL = discover_generator_snapshot()

print("train:", TRAIN_PATH)
print("public:", PUBLIC_PATH)
print("selected-contexts:", CONTEXTS_PATH or "không có — generation-only không cần retrieval")
print("source submission:", SOURCE_SUBMISSION_PATH or "không có — dùng đáp án train")
print("comparison submission:", COMPARISON_SUBMISSION_PATH or "không có")
print("generator:", GENERATOR_MODEL)


Không thấy snapshot; sẽ tải generator từ Hugging Face.
train: /kaggle/input/datasets/lighth/uit-dsc-2026-task2-legalqa-train/train.json
public: /kaggle/input/datasets/lighth/uit-dsc-2026-task2-legalqa-train/public-official.json
selected-contexts: /kaggle/input/datasets/lighth/uit-dsc-2026-task2-legalqa-train/selected-contexts/selected-contexts
source submission: không có — dùng đáp án train
comparison submission: không có
generator: AITeamVN/Vi-Qwen2-1.5B-RAG


## 3. Chọn các đáp án dài để kiểm tra

Notebook ưu tiên `submission_v0.json` nếu tự tìm thấy hoặc được chỉ định qua `SOURCE_SUBMISSION_PATH`; câu hỏi tương ứng lấy từ public test. Nếu không có submission nguồn, notebook dùng câu hỏi/đáp án từ `train.json`. Khi có comparison submission, các ID bị rút ngắn mạnh được ưu tiên.


In [4]:
def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


train_data = load_json(TRAIN_PATH)
public_data = load_json(PUBLIC_PATH)
source_submission = load_json(SOURCE_SUBMISSION_PATH) if SOURCE_SUBMISSION_PATH else {}
comparison_data = load_json(COMPARISON_SUBMISSION_PATH) if COMPARISON_SUBMISSION_PATH else {}

assert isinstance(train_data, dict) and train_data, "train.json phải là JSON object không rỗng"
assert isinstance(public_data, dict) and public_data, "public test phải là JSON object không rỗng"

if source_submission:
    assert all(
        isinstance(value, dict) and isinstance(value.get("answer"), str)
        for value in source_submission.values()
    ), "source submission phải có dạng {id: {'answer': str}}"
    source_data = source_submission
    question_data = public_data
    source_label = str(SOURCE_SUBMISSION_PATH)
else:
    source_data = {
        str(qid): {"answer": str(item["answer"])}
        for qid, item in train_data.items()
        if isinstance(item, dict) and isinstance(item.get("answer"), str)
    }
    question_data = train_data
    source_label = str(TRAIN_PATH)

questions = {
    str(qid): str(item.get("question") or "")
    for qid, item in question_data.items()
    if isinstance(item, dict)
}
assert source_data, "Không có answer nguồn để chạy smoke test"

common_ids = set(source_data) & set(comparison_data) if comparison_data else set(source_data)
if comparison_data:
    ranked_ids = sorted(
        common_ids,
        key=lambda qid: len(source_data[qid]["answer"]) - len(comparison_data[qid].get("answer", "")),
        reverse=True,
    )
else:
    ranked_ids = sorted(common_ids, key=lambda qid: len(source_data[qid]["answer"]), reverse=True)

selected_ids = []
for qid in PRIORITY_IDS + ranked_ids:
    if qid in common_ids and qid not in selected_ids:
        selected_ids.append(qid)
    if len(selected_ids) >= TEST_LIMIT:
        break

selection_rows = []
for qid in selected_ids:
    source = source_data[qid]["answer"]
    comparison = comparison_data.get(qid, {}).get("answer", "")
    selection_rows.append({
        "id": qid,
        "question": questions.get(qid, ""),
        "source_chars": len(source),
        "comparison_chars": len(comparison) if comparison else None,
        "shrink_chars": len(source) - len(comparison) if comparison else None,
    })

selection_df = pd.DataFrame(selection_rows)
display(selection_df)
print(f"Nguồn context: {source_label}")
print(f"Đã chọn {len(selected_ids)} ID: {selected_ids}")


,id,question,source_chars,comparison_chars,shrink_chars
0,37801,Thủ tục phê duyệt và cấp bảo lãnh Chính phủ đố...,10755,None,None
1,118227,Thời điểm lập hóa đơn thuế GTGT năm 2023 được ...,9719,None,None
2,79987,Vay ưu đãi để thuê mua nhà ở xã hội theo thủ t...,9128,None,None
3,113337,"Quản lý, giám sát hoạt động của tàu cá giữa bi...",8089,None,None
4,131625,Bản đồ địa chính theo Luật đất đai thể hiện nh...,8082,None,None
5,120573,Các cơ sở kinh doanh dịch vụ karaoke trong mỗi...,7844,None,None
6,113663,Đơn đăng ký bảo hộ logo công ty phải đáp ứng n...,7674,None,None
7,12581,"Trình tự, thủ tục chuyển nhượng hợp đồng mua b...",7568,None,None
8,61739,Hướng dẫn về nội dung và phương pháp lập các c...,7498,None,None
9,164923,Mặt hàng cẩu trục có chịu thuế giá trị gia tăn...,6963,None,None


Nguồn context: /kaggle/input/datasets/lighth/uit-dsc-2026-task2-legalqa-train/train.json
Đã chọn 10 ID: ['37801', '118227', '79987', '113337', '131625', '120573', '113663', '12581', '61739', '164923']


## 4. Load riêng Vi-Qwen2-1.5B-RAG

Cell này là bước nặng duy nhất. Không load embedding model hoặc reranker.


In [5]:
set_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL,
    trust_remote_code=False,
)

load_kwargs = {
    "trust_remote_code": False,
    "low_cpu_mem_usage": True,
}
if torch.cuda.is_available():
    load_kwargs.update({
        "device_map": "auto",
        "torch_dtype": torch.float16,
    })
else:
    load_kwargs["torch_dtype"] = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    **load_kwargs,
)
model.eval()

MODEL_DEVICE = next(model.parameters()).device
prompt_helper = ViQwenRAGGenerator(
    model_name_or_path=GENERATOR_MODEL,
    max_new_tokens=512,
    max_input_tokens=MAX_INPUT_TOKENS_512,
    repetition_penalty=REPETITION_PENALTY,
    seed=SEED,
)
prompt_helper._tokenizer = tokenizer
prompt_helper._model = model

print("Model device:", MODEL_DEVICE)
print("Model loaded:", GENERATOR_MODEL)


config.json:   0%|          | 0.00/709 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model device: cuda:0
Model loaded: AITeamVN/Vi-Qwen2-1.5B-RAG


## 5. Prompt chống tóm tắt và hàm audit


In [6]:
GENERIC_QUESTION = (
    "Hãy trả lại đầy đủ nội dung pháp lý cần thiết trong CONTEXT. "
    "Không tóm tắt, không lược bỏ danh sách, điều khoản hoặc biểu mẫu."
)


@torch.inference_mode()
def generate_one(question, context, max_input_tokens, max_new_tokens):
    model_context = int(
        getattr(model.config, "max_position_embeddings", max_input_tokens)
    )
    if max_new_tokens >= model_context:
        raise ValueError(
            f"max_new_tokens={max_new_tokens} phải nhỏ hơn context window={model_context}"
        )
    prompt_limit = min(max_input_tokens, model_context - max_new_tokens)
    inputs = prompt_helper._prepare_inputs(
        context=context,
        question=question or GENERIC_QUESTION,
        prompt_limit=prompt_limit,
    )
    inputs = {key: value.to(MODEL_DEVICE) for key, value in inputs.items()}
    input_length = inputs["input_ids"].shape[1]

    generate_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": False,
        "repetition_penalty": REPETITION_PENALTY,
    }
    eos_token_id, pad_token_id = prompt_helper._generation_token_ids()
    if eos_token_id is not None:
        generate_kwargs["eos_token_id"] = eos_token_id
    if pad_token_id is not None:
        generate_kwargs["pad_token_id"] = pad_token_id

    started = time.time()
    outputs = model.generate(
        **inputs,
        **generate_kwargs,
    )
    elapsed = time.time() - started

    generated_ids = outputs[0, input_length:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return {
        "answer": answer,
        "input_tokens": int(input_length),
        "generated_tokens": int(generated_ids.shape[0]),
        "hit_token_limit": bool(generated_ids.shape[0] >= max_new_tokens - 4),
        "seconds": round(elapsed, 2),
    }


def word_tokens(text):
    return re.findall(r"\w+", text.lower(), flags=re.UNICODE)


def source_coverage(source, prediction):
    src = Counter(word_tokens(source))
    pred = Counter(word_tokens(prediction))
    if not src:
        return 0.0
    return sum((src & pred).values()) / sum(src.values())


def audit(source, generated):
    answer = generated["answer"].strip()
    lower = answer.lower()
    source_words = len(word_tokens(source))
    answer_words = len(word_tokens(answer))
    possibly_cut = bool(
        len(answer) > 100
        and not answer.endswith((".", "!", "?", ")", "]", "}", '"', "”"))
    )
    result = {
        **generated,
        "source_words": source_words,
        "answer_words": answer_words,
        "length_ratio": round(answer_words / max(1, source_words), 3),
        "source_coverage": round(source_coverage(source, answer), 3),
        "has_boilerplate": bool("dựa trên ngữ cảnh" in lower or "theo ngữ cảnh" in lower),
        "says_no_information": bool("không đủ thông tin" in lower or "không có thông tin" in lower),
        "possibly_cut": possibly_cut,
    }
    result["passes_smoke_test"] = bool(
        not result["hit_token_limit"]
        and not result["has_boilerplate"]
        and not result["says_no_information"]
        and not result["possibly_cut"]
        and result["length_ratio"] >= 0.70
        and result["source_coverage"] >= 0.65
    )
    return result


## 6. Chạy smoke test 512 và 1024 token

Mỗi ID dùng đúng cùng một source context. Vì vậy khác biệt ở kết quả đến từ prompt/generation, không phải retrieval.


In [7]:
configs = [
    ("strict_512", MAX_INPUT_TOKENS_512, 512),
]
if RUN_1024_TEST:
    configs.append(("strict_1024", MAX_INPUT_TOKENS_1024, 1024))

all_results = []
for item_index, qid in enumerate(selected_ids, start=1):
    context = source_data[qid]["answer"]
    question = (
        questions.get(qid) or GENERIC_QUESTION
        if USE_ACTUAL_QUESTIONS
        else GENERIC_QUESTION
    )
    print(f"\n[{item_index}/{len(selected_ids)}] ID={qid} | context={len(word_tokens(context))} words")

    for config_name, max_input_tokens, max_new_tokens in configs:
        generated = generate_one(
            question=question,
            context=context,
            max_input_tokens=max_input_tokens,
            max_new_tokens=max_new_tokens,
        )
        checked = audit(context, generated)
        record = {
            "id": qid,
            "config": config_name,
            "question": question,
            **checked,
        }
        all_results.append(record)
        print(
            f"  {config_name}: words={checked['answer_words']} "
            f"ratio={checked['length_ratio']:.3f} coverage={checked['source_coverage']:.3f} "
            f"tokens={checked['generated_tokens']} hit_limit={checked['hit_token_limit']} "
            f"pass={checked['passes_smoke_test']} ({checked['seconds']}s)"
        )

RESULT_JSON = EXPORT_DIR / "legalqa_generation_smoke_test_results.json"
RESULT_CSV = EXPORT_DIR / "legalqa_generation_smoke_test_summary.csv"

RESULT_JSON.write_text(
    json.dumps(all_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

summary_columns = [
    "id", "config", "source_words", "answer_words", "length_ratio",
    "source_coverage", "generated_tokens", "hit_token_limit",
    "has_boilerplate", "says_no_information", "possibly_cut",
    "passes_smoke_test", "seconds",
]
result_df = pd.DataFrame(all_results)
result_df[summary_columns].to_csv(RESULT_CSV, index=False, encoding="utf-8-sig")

print("\nSaved:", RESULT_JSON)
print("Saved:", RESULT_CSV)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



[1/10] ID=37801 | context=2401 words
  strict_512: words=382 ratio=0.159 coverage=0.156 tokens=512 hit_limit=True pass=False (28.05s)
  strict_1024: words=532 ratio=0.222 coverage=0.214 tokens=692 hit_limit=False pass=False (35.17s)

[2/10] ID=118227 | context=2137 words
  strict_512: words=141 ratio=0.066 coverage=0.058 tokens=177 hit_limit=False pass=False (9.89s)
  strict_1024: words=141 ratio=0.066 coverage=0.058 tokens=177 hit_limit=False pass=False (9.85s)

[3/10] ID=79987 | context=2096 words
  strict_512: words=369 ratio=0.176 coverage=0.173 tokens=512 hit_limit=True pass=False (26.19s)
  strict_1024: words=746 ratio=0.356 coverage=0.352 tokens=1024 hit_limit=True pass=False (50.15s)

[4/10] ID=113337 | context=1826 words
  strict_512: words=380 ratio=0.208 coverage=0.208 tokens=512 hit_limit=True pass=False (25.85s)
  strict_1024: words=801 ratio=0.439 coverage=0.438 tokens=1024 hit_limit=True pass=False (50.12s)

[5/10] ID=131625 | context=1763 words
  strict_512: words=375 

## 7. Kết luận nhanh

- `hit_token_limit=True`: ngân sách output chưa đủ; câu có khả năng bị cắt.
- `length_ratio < 0.70` hoặc `source_coverage < 0.65`: model vẫn rút quá mạnh với câu dài.
- Nếu 1024 vẫn thất bại, không nên bắt model sinh lại toàn bộ; submission nên trả raw chunk tốt nhất + chunk liền kề.


In [8]:
summary_columns = [
    "id", "config", "source_words", "answer_words", "length_ratio",
    "source_coverage", "generated_tokens", "hit_token_limit",
    "has_boilerplate", "says_no_information", "possibly_cut",
    "passes_smoke_test", "seconds",
]

display(result_df[summary_columns].sort_values(["config", "passes_smoke_test", "length_ratio"]))

aggregate = result_df.groupby("config").agg(
    samples=("id", "count"),
    pass_rate=("passes_smoke_test", "mean"),
    mean_length_ratio=("length_ratio", "mean"),
    mean_source_coverage=("source_coverage", "mean"),
    hit_limit_rate=("hit_token_limit", "mean"),
    boilerplate_rate=("has_boilerplate", "mean"),
    no_information_rate=("says_no_information", "mean"),
    mean_seconds=("seconds", "mean"),
).reset_index()
display(aggregate)

best_config = aggregate.sort_values(
    ["pass_rate", "mean_source_coverage", "mean_length_ratio"],
    ascending=False,
).iloc[0]

print(f"Cấu hình tốt nhất trong smoke test: {best_config['config']}")
print(f"Pass rate: {best_config['pass_rate']:.1%}")

if best_config["pass_rate"] < 0.80:
    print("KẾT LUẬN: Generation vẫn không ổn định với đáp án dài. Hãy dùng raw context + adjacent chunks làm fallback khi submission.")
else:
    print("KẾT LUẬN: Prompt/token budget đã đủ tốt trên mẫu dài. Tiếp tục test thêm 20–50 câu trước submission.")

display(FileLink(str(RESULT_JSON)))
display(FileLink(str(RESULT_CSV)))


,id,config,source_words,answer_words,length_ratio,source_coverage,generated_tokens,hit_token_limit,has_boilerplate,says_no_information,possibly_cut,passes_smoke_test,seconds
11,120573,strict_1024,1734,73,0.042,0.039,104,False,False,False,False,False,6.02
3,118227,strict_1024,2137,141,0.066,0.058,177,False,True,True,False,False,9.85
19,164923,strict_1024,1487,158,0.106,0.075,214,False,False,True,False,False,11.09
1,37801,strict_1024,2401,532,0.222,0.214,692,False,True,False,False,False,35.17
5,79987,strict_1024,2096,746,0.356,0.352,1024,True,True,False,True,False,50.15
13,113663,strict_1024,1745,690,0.395,0.382,925,False,True,False,False,False,45.92
9,131625,strict_1024,1763,768,0.436,0.417,1024,True,False,False,True,False,49.09
7,113337,strict_1024,1826,801,0.439,0.438,1024,True,False,False,True,False,50.12
17,61739,strict_1024,1625,726,0.447,0.445,1024,True,True,False,True,False,50.00
15,12581,strict_1024,1674,798,0.477,0.475,1024,True,False,False,True,False,50.16


,config,samples,pass_rate,mean_length_ratio,mean_source_coverage,hit_limit_rate,boilerplate_rate,no_information_rate,mean_seconds
0,strict_1024,10,0.0,0.2986,0.2895,0.5,0.5,0.2,35.757
1,strict_512,10,0.0,0.1622,0.1552,0.7,0.5,0.2,21.067


Cấu hình tốt nhất trong smoke test: strict_1024
Pass rate: 0.0%
KẾT LUẬN: Generation vẫn không ổn định với đáp án dài. Hãy dùng raw context + adjacent chunks làm fallback khi submission.


/kaggle/working/legalqa_generation_smoke_test_results.json

/kaggle/working/legalqa_generation_smoke_test_summary.csv

## 8. Xem chi tiết một vài câu


In [9]:
for qid in selected_ids[:3]:
    display(Markdown(f"### ID {qid}"))
    question = (
        questions.get(qid) or GENERIC_QUESTION
        if USE_ACTUAL_QUESTIONS
        else GENERIC_QUESTION
    )
    print("QUESTION:", question)
    print("\nSOURCE CONTEXT:\n", source_data[qid]["answer"])
    for record in all_results:
        if record["id"] == qid:
            print(f"\n--- {record['config']} ---")
            print(record["answer"])


### ID 37801

QUESTION: Thủ tục phê duyệt và cấp bảo lãnh Chính phủ đối với khoản vay nước ngoài của doanh nghiệp

SOURCE CONTEXT:
 (1) Hồ sơ đề nghị phê duyệt cấp bảo lãnh chính phủ đối với khoản vay theo Điều 14 Nghị định 91/2018/NĐ-CP:
Ngoài hồ sơ đã gửi theo quy định tại Điều 11 Nghị định này, người vay đề nghị phê duyệt cấp bảo lãnh chính phủ đối với khoản vay nộp cập nhật cho Bộ Tài chính trực tiếp hoặc thông qua dịch vụ bưu chính các hồ sơ sau:
- Văn bản yêu cầu khoản vay có bảo lãnh chính phủ của người cho vay gửi người vay (bản chính).
- Văn bản đề nghị cấp bảo lãnh chính phủ của doanh nghiệp kèm theo đề xuất ngân hàng phục vụ cho khoản vay được Chính phủ bảo lãnh (bản chính).
- Các văn bản theo quy định tại Điều 11 Nghị định này nếu có bất kỳ điều chỉnh nào so với văn bản đã nộp trước đây.
- Báo cáo nghiên cứu khả thi đã được cấp có thẩm quyền phê duyệt theo quy định của pháp luật về đầu tư và đầu tư công (trường hợp nộp cho Bộ Tài chính báo cáo nghiên cứu tiền khả thi quy định tại điểm a 

### ID 118227

QUESTION: Thời điểm lập hóa đơn thuế GTGT năm 2023 được xác định như thế nào?

SOURCE CONTEXT:
 Căn cứ khoản 1, khoản 2 Điều 9 Nghị định 123/2020/NĐ-CP, thời điểm lập hóa đơn đối với bán hàng hóa và cung cấp dịch vụ được quy định rõ như sau:
Thời điểm xuất hóa đơn khi bán hàng hóa
Thời điểm lập hóa đơn đối với bán hàng hóa (bao gồm cả bán tài sản nhà nước, tài sản tịch thu, sung quỹ nhà nước và bán hàng dự trữ quốc gia) là thời điểm chuyển giao quyền sở hữu hoặc quyền sử dụng hàng hóa cho người mua, không phân biệt đã thu được tiền hay chưa thu được tiền.
Thời điểm xuất hóa đơn khi cung cấp dịch vụ
Thời điểm lập hóa đơn đối với cung cấp dịch vụ là thời điểm hoàn thành việc cung cấp dịch vụ không phân biệt đã thu được tiền hay chưa thu được tiền.
Trường hợp người cung cấp dịch vụ có thu tiền trước hoặc trong khi cung cấp dịch vụ thì thời điểm lập hóa đơn là thời điểm thu tiền (không bao gồm trường hợp thu tiền đặt cọc hoặc tạm ứng để đảm bảo thực hiện hợp đồng cung cấp các dịch vụ: Kế t

### ID 79987

QUESTION: Vay ưu đãi để thuê mua nhà ở xã hội theo thủ tục nào?

SOURCE CONTEXT:
 Thủ tục vay ưu đãi để thuê mua nhà ở xã hội đối với công chức Nhà nước thực hiện theo khoản 8 và khoản 9 Hướng dẫn 2526/NHCS-TDSV năm 2016 như sau:
* Hồ sơ vay vốn
- Giấy đề nghị vay vốn theo mẫu số 01/NƠXH;
- Giấy xác nhận về đối tượng và thực trạng nhà ở: Giấy xác nhận theo mẫu số 03 tại Phụ lục I ban hành kèm theo Thông tư 20/2016/TT- BXD;
- Giấy chứng minh về điều kiện thu nhập:
+ Việc xác nhận về điều kiện thu nhập thực hiện đồng thời với việc xác nhận đối tượng và thực trạng nhà ở áp dụng theo mẫu số 03 tại Phụ lục I ban hành kèm theo Thông tư 20/2016/TT-BXD;
+ Trường hợp công chức đã được xác nhận về đối tượng và điều kiện để được hưởng chính sách hỗ trợ nhà ở xã hội trước ngày 15/8/2016, nhưng chưa có xác nhận về điều kiện thu nhập thì phải xác nhận bổ sung về điều kiện thu nhập theo mẫu số 07 tại Phụ lục I ban hành kèm theo Thông tư 20/2016/TT-BXD.
+ Trường hợp công chức đã nghỉ việc, nghỉ chế độ